# Election Bloc Change Prediction Project
## Notebook 06 — Unseen localities and frozen candidate

Uses grouped cross-validation over the full development period, compares the selected historical Ridge model against persistence, and freezes a candidate before opening K24→K25.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 300)
REPO_URL = "https://github.com/IfatDav/Election_Bloc_Prediction_Project.git"
DEFAULT_REPO = Path("/content/Election_Bloc_Prediction_Project")

def locate_repo():
    candidates=[]
    if os.getenv("ELECTION_PROJECT_ROOT"):
        candidates.append(Path(os.environ["ELECTION_PROJECT_ROOT"]).expanduser())
    cwd=Path.cwd().resolve()
    candidates += [cwd, *cwd.parents, DEFAULT_REPO]
    for p in candidates:
        if (p/"data"/"raw").exists(): return p
    if Path("/content").exists():
        if DEFAULT_REPO.exists(): shutil.rmtree(DEFAULT_REPO)
        subprocess.run(["git","clone","--depth","1",REPO_URL,str(DEFAULT_REPO)],check=True)
        return DEFAULT_REPO
    raise FileNotFoundError("Repository not found. Set ELECTION_PROJECT_ROOT.")

REPO_ROOT=locate_repo()
RAW_DIR=REPO_ROOT/"data"/"raw"
INTERIM_DIR=REPO_ROOT/"data"/"interim"
PROCESSED_DIR=REPO_ROOT/"data"/"processed"
REPORTS_DIR=REPO_ROOT/"reports"
TABLES_DIR=REPORTS_DIR/"tables"
FIGURES_DIR=REPORTS_DIR/"figures"
SUMMARIES_DIR=REPORTS_DIR/"summaries"
MODELS_DIR=REPO_ROOT/"models"
for d in [INTERIM_DIR,PROCESSED_DIR,TABLES_DIR,FIGURES_DIR,SUMMARIES_DIR,MODELS_DIR]: d.mkdir(parents=True,exist_ok=True)
print("Repository:",REPO_ROOT)

In [ ]:
import joblib
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
DATA=INTERIM_DIR/"modeling_transition_features.csv"; ART=MODELS_DIR/"best_historical_delta_model.joblib"; SEL=SUMMARIES_DIR/"selected_historical_delta_model.json"
if not DATA.exists() and (REPO_ROOT/".git").exists(): subprocess.run(["git","-C",str(REPO_ROOT),"pull","--ff-only","origin","main"],check=True)
df=pd.read_csv(DATA,dtype={"locality_symbol":"string"},low_memory=False);artifact=joblib.load(ART);selection=json.loads(SEL.read_text(encoding="utf-8-sig"))
B=artifact["bloc_order"];TARGET=artifact["target_columns"];COLS=artifact["feature_columns"];ALPHA=float(artifact["alpha"]);PREVCLR=[f"previous_clr_{b}" for b in B];PREVPCT=[f"previous_{b}_pct" for b in B];CURRPCT=[f"current_{b}_pct" for b in B]
DEV=[f"K{n}_to_K{n+1}" for n in range(16,24)];dev=df[df.transition_id.isin(DEV)&df.has_current_core_features].copy().reset_index(drop=True)


def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return (series.astype("string").str.strip().str.lower()
            .map({"true":True,"false":False,"1":True,"0":False,"yes":True,"no":False})
            .fillna(False).astype(bool))

df["has_current_core_features"] = to_bool(df["has_current_core_features"])

print(len(dev),dev.locality_symbol.nunique(),selection)

## Grouped out-of-fold predictions

In [ ]:
def onehot():
    try:return OneHotEncoder(handle_unknown="ignore",sparse_output=False)
    except TypeError:return OneHotEncoder(handle_unknown="ignore",sparse=False)
def prep(data,cols):
    x=data[cols].copy()
    for c in cols:
        if c in {"locality_type","analysis_group"}:x[c]=x[c].astype("string").fillna("Unknown").astype(str)
        else:x[c]=pd.to_numeric(x[c],errors="coerce")
    return x
def pipeline(cols):
    cat=[c for c in cols if c in {"locality_type","analysis_group"}];num=[c for c in cols if c not in cat];trs=[]
    if num:trs.append(("num",Pipeline([("imp",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("scale",StandardScaler())]),num))
    if cat:trs.append(("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("oh",onehot())]),cat))
    return Pipeline([("pre",ColumnTransformer(trs)),("model",Ridge(alpha=ALPHA))])
def center(z):z=np.asarray(z,float);return z-z.mean(1,keepdims=True)
def invclr(z):e=np.exp(z-z.max(1,keepdims=True));return e/e.sum(1,keepdims=True)*100
def to_pct(data,delta):return invclr(data[PREVCLR].to_numpy(float)+center(delta))
def evaluate(data,pred,name):
    y=data[CURRPCT].to_numpy(float);ae=np.abs(y-pred);row=ae.mean(1);w=pd.to_numeric(data.current_valid_votes,errors="coerce").fillna(0).to_numpy();return {"strategy":name,"rows":len(data),"mae":float(ae.mean()),"median_row_mae":float(np.median(row)),"weighted_mae":float(np.average(row,weights=w)) if w.sum()>0 else np.nan}

groups=dev.locality_symbol.astype(str);splitter=GroupKFold(n_splits=5);oof=np.full((len(dev),4),np.nan);fold=np.zeros(len(dev),int);aud=[]
for k,(tr,te) in enumerate(splitter.split(dev,groups=groups),1):
    m=pipeline(COLS).fit(prep(dev.iloc[tr],COLS),dev.iloc[tr][TARGET].to_numpy(float));oof[te]=center(m.predict(prep(dev.iloc[te],COLS)));fold[te]=k
    overlap=set(groups.iloc[tr])&set(groups.iloc[te]);aud.append({"fold":k,"train_rows":len(tr),"test_rows":len(te),"train_localities":groups.iloc[tr].nunique(),"test_localities":groups.iloc[te].nunique(),"locality_overlap":len(overlap)})
if np.isnan(oof).any() or sum(a["locality_overlap"] for a in aud):raise ValueError("Grouped CV failed")

## Shrinkage and group diagnostics

In [ ]:
LAMBDAS=np.round(np.linspace(0,1,21),2); shrink=[]
for lam in LAMBDAS: shrink.append({"lambda":lam,**evaluate(dev,to_pct(dev,oof*lam),f"lambda={lam}")})
shrink=pd.DataFrame(shrink);best_lambda=float(shrink.sort_values(["mae","lambda"]).iloc[0]["lambda"])
preds={"Previous-election persistence":dev[PREVPCT].to_numpy(float),"Ridge — raw correction":to_pct(dev,oof),"Ridge — shrunk correction":to_pct(dev,oof*best_lambda)}
overall=[];groups_out=[];blocs=[]
for name,p in preds.items():
    overall.append(evaluate(dev,p,name));y=dev[CURRPCT].to_numpy(float)
    for g,sub in dev.groupby("analysis_group"):
        loc=dev.index.get_indexer(sub.index);groups_out.append({**evaluate(sub,p[loc],name),"analysis_group":g})
    for j,b in enumerate(B):blocs.append({"strategy":name,"bloc":b,"rows":len(dev),"mae":float(np.abs(y[:,j]-p[:,j]).mean())})
overall=pd.DataFrame(overall).sort_values("mae");overall

## Freeze candidate using temporal and unseen-locality evidence

In [ ]:
unseen_p=float(overall.loc[overall.strategy.eq("Previous-election persistence"),"mae"].iloc[0]);unseen_r=float(overall.loc[overall.strategy.eq("Ridge — shrunk correction"),"mae"].iloc[0])
temporal_p=float(selection["persistence_mean_temporal_mae"]);temporal_r=float(selection["mean_temporal_mae"])
if unseen_r<unseen_p and temporal_r<temporal_p:
    frozen="Ridge — shrunk correction";reason="Selected learned architecture beats persistence in both rolling temporal backtests and unseen-locality grouped CV."
else:
    frozen="Previous-election persistence";reason="No learned strategy beats persistence in both required validation dimensions."
summary={"notebook":"06_unseen_localities","selected_model":artifact["model_name"],"alpha":ALPHA,"shrinkage_lambda":best_lambda,"unseen_locality_comparison":overall.to_dict("records"),"temporal_persistence_mae":temporal_p,"temporal_learned_mae":temporal_r,"frozen_candidate":frozen,"reason":reason,"final_test_status":"K24_to_K25 remains locked"}
paths={"comparison":TABLES_DIR/"unseen_locality_cv_comparison.csv","group":TABLES_DIR/"unseen_locality_cv_by_group.csv","bloc":TABLES_DIR/"unseen_locality_cv_by_bloc.csv","fold":TABLES_DIR/"unseen_locality_cv_fold_audit.csv","shrink":TABLES_DIR/"unseen_locality_shrinkage_selection.csv","pred":PROCESSED_DIR/"unseen_locality_oof_predictions.csv","candidate":SUMMARIES_DIR/"final_candidate_before_test.json","summary":SUMMARIES_DIR/"notebook_06_summary.json"}
overall.to_csv(paths["comparison"],index=False,encoding="utf-8-sig");pd.DataFrame(groups_out).to_csv(paths["group"],index=False,encoding="utf-8-sig");pd.DataFrame(blocs).to_csv(paths["bloc"],index=False,encoding="utf-8-sig");pd.DataFrame(aud).to_csv(paths["fold"],index=False,encoding="utf-8-sig");shrink.to_csv(paths["shrink"],index=False,encoding="utf-8-sig")
out=dev[["locality_symbol","locality_name","transition_id","locality_type","analysis_group",*PREVPCT,*CURRPCT]].copy();out["grouped_cv_fold"]=fold
for name,p in preds.items():
    safe=re.sub(r"[^a-z0-9]+","_",name.lower()).strip("_")
    for j,b in enumerate(B):out[f"{safe}__predicted_{b}_pct"]=p[:,j]
out.to_csv(paths["pred"],index=False,encoding="utf-8-sig");paths["candidate"].write_text(json.dumps({"frozen_candidate":frozen,"reason":reason,"shrinkage_lambda":best_lambda,"final_test_status":"locked"},ensure_ascii=False,indent=2),encoding="utf-8");paths["summary"].write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(summary,ensure_ascii=False,indent=2))